# Isentropic streamfunction: $\theta_e$ versus frozen $\theta_e$

Compares the isentropic streamfunction computed with two vertical coordinates:

- $\theta_e$: `thermo.equivalent_potential_temperature` (liquid water reference, Emanuel 1994 with ice),
- $\theta_{ef}$: `thermo.frozen_equivalent_potential_temperature` (ice reference, Pauluis 2018).

Both are computed from the same snapshot, so each file is read only once. The conditional sums
keep the experiment and time dependence, as in `isentropic_streamfunction.ipynb`.

Use the **Python (iso)** kernel.

## Configuration

In [ ]:
from pathlib import Path

DATA_DIR = Path("/scratch/ttg2015/mailbox/OPaul/2026-10-23_vddgrid_3d_30Days_Perpetual")
PATTERN = "*.nc"

# SAM 6.11 3D snapshot output
#   W (m/s), TABS (K), QV, QN (g/kg), p(z) (mb), PP (Pa)
VAR = dict(w="W", T="TABS", p="p", pp="PP", qv="QV", qn="QN")
Z_DIM = "z"
TIME_DIM = "time"
T_ICE, T_LIQ = 253.16, 273.16   # SAM's partition of QN into ice and liquid

# Times to process, e.g. slice(120, None, 6) for a quick test on every 6th snapshot of the last half
TIME_SLICE = slice(None)

# Bins (K) for each coordinate. The frozen theta_e is higher than theta_e for moist air
# (vapor carries the latent heat of sublimation), so check the ranges printed below and adjust.
BINS = {
    "theta_e":  dict(th_min=300.0, th_max=370.0, nbin=141),
    "theta_ef": dict(th_min=300.0, th_max=390.0, nbin=181),
}
LABEL = {"theta_e": r"$\theta_e$", "theta_ef": r"$\theta_{ef}$ (frozen)"}

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import isentropic as iso
from isentropic import thermo

files = sorted(DATA_DIR.glob(PATTERN))
print(f"{len(files)} files")
ds0 = xr.open_dataset(files[0])
z = ds0[Z_DIM].values

## Thermodynamic fields at one time

In [ ]:
def load_time(ds, it):
    """Return rho*w, theta_e, frozen theta_e (nz, ny, nx) and the density profile (nz) at time index it."""
    snap = ds.isel({TIME_DIM: it})
    get = lambda name: snap[name].values.astype(float)
    T = get(VAR["T"])
    qv = np.maximum(get(VAR["qv"]) * 1e-3, 1e-10)           # g/kg -> kg/kg
    qn = np.maximum(get(VAR["qn"]) * 1e-3, 0.0)
    liq_frac = np.clip((T - T_ICE) / (T_LIQ - T_ICE), 0.0, 1.0)
    ql, qi = qn * liq_frac, qn * (1.0 - liq_frac)
    p = get(VAR["p"])[:, None, None] * 100.0 + get(VAR["pp"])   # mb -> Pa, plus perturbation
    coords = {
        "theta_e": thermo.equivalent_potential_temperature(p, T, qv, ql, qi),
        "theta_ef": thermo.frozen_equivalent_potential_temperature(p, T, qv, ql, qi),
    }
    rho = thermo.density(p, T, qv, ql, qi).mean((1, 2))        # density profile (anelastic)
    return rho[:, None, None] * get(VAR["w"]), coords, rho


rhow, coords, rho = load_time(ds0, 0)
for name, th in coords.items():
    print(f"{name:8s} range: {np.nanmin(th):.1f} - {np.nanmax(th):.1f} K   bins: {BINS[name]}")

## Conditional sums for each experiment and time
The result has one set of variables per coordinate: `MF_theta_e(experiment, time, z, theta_e)`,
`MF_theta_ef(experiment, time, z, theta_ef)`, and the same for `AREA` and `ATH` (sum of the coordinate itself).

In [ ]:
centers = {name: b["th_min"] + (b["th_max"] - b["th_min"]) / (b["nbin"] - 1) * np.arange(b["nbin"])
           for name, b in BINS.items()}
dth = {name: c[1] - c[0] for name, c in centers.items()}


def process_experiment(f):
    """Conditional sums in both coordinates at every selected time of one experiment file."""
    with xr.open_dataset(f) as ds:
        times = np.arange(ds.sizes[TIME_DIM])[TIME_SLICE]
        out = {}
        for name, b in BINS.items():
            for v in ("MF", "AREA", "ATH"):
                out[f"{v}_{name}"] = np.zeros((len(times), len(z), b["nbin"]))
        RHO = np.zeros((len(times), len(z)))
        for k, it in enumerate(times):
            rhow, coords, RHO[k] = load_time(ds, it)
            for name, th in coords.items():
                mask = ~np.isfinite(th)
                (mf, ath), area = iso.iso_multi_integral(np.stack([rhow, th]), th, mask, 1.0, **BINS[name])
                out[f"MF_{name}"][k], out[f"ATH_{name}"][k], out[f"AREA_{name}"][k] = mf, ath, area
            if k % 24 == 0:
                print(f.stem, "time", it, "/", ds.sizes[TIME_DIM])
        data = {key: (("time", "z", key.split("_", 1)[1]), val) for key, val in out.items()}
        data["RHO"] = (("time", "z"), RHO)
        return xr.Dataset(data, coords={"time": ds[TIME_DIM].values[times], "z": z, **centers}
                          ).expand_dims(experiment=[f.stem])


sums = xr.concat([process_experiment(f) for f in files], dim="experiment", join="outer")
sums

## Streamfunctions
$\Psi$ is computed for every experiment and time in each coordinate, at the upper edge of each bin.

In [ ]:
result = sums.copy()
for name in BINS:
    mf, area = sums[f"MF_{name}"].values, sums[f"AREA_{name}"].values
    ts = iso.two_stream(mf, area)
    result[f"PSI_{name}"] = (("experiment", "time", "z", name), iso.streamfunction(mf, area))
    result[f"PSI_{name}"].attrs.update(long_name=f"isentropic streamfunction in {name}", units="kg m-2 s-1",
                                       note=f"evaluated at {name} + d{name}/2")
    result[f"MEAN_{name}"] = sums[f"ATH_{name}"].sum(name) / sums[f"AREA_{name}"].sum(name)
    result[f"M_UP_{name}"] = (("experiment", "time", "z"), ts["M_up"])
    result[f"M_DOWN_{name}"] = (("experiment", "time", "z"), ts["M_down"])
result.attrs.update(source=str(DATA_DIR), **{f"d{name}": d for name, d in dth.items()})
result

## Comparison, one figure per experiment
Top: time-mean $\Psi$ in $\theta_e$ and in frozen $\theta_e$, with the same colour scale and the mean
profile of each coordinate. Bottom: the two-stream ascending and descending mass fluxes $M^\pm$, and the
overturning $\max_\theta |\Psi|$ in each coordinate. Both top panels use the same $\theta$ axis.

In [ ]:
# Time means: sum the conditional sums over time, then compute Psi
tm = {}
for name in BINS:
    mf = sums[f"MF_{name}"].sum("time", skipna=True).values
    area = sums[f"AREA_{name}"].sum("time", skipna=True).values
    ts = iso.two_stream(mf, area)
    tm[name] = dict(psi=iso.streamfunction(mf, area),
                    mean=sums[f"ATH_{name}"].sum("time", skipna=True).sum(name).values / area.sum(-1),
                    M_up=ts["M_up"], M_down=ts["M_down"])
lim = max(np.nanmax(np.abs(t["psi"])) for t in tm.values())
levels = np.linspace(-lim, lim, 20)
xlim = (min(b["th_min"] for b in BINS.values()), max(b["th_max"] for b in BINS.values()))

for e, exp in enumerate(sums.experiment.values):
    fig, axs = plt.subplots(2, 2, figsize=(12, 9), sharey=True)
    for ax, name in zip(axs[0], BINS):
        th = centers[name] + dth[name] / 2
        pc = ax.pcolormesh(th, z / 1e3, tm[name]["psi"][e], cmap="RdBu_r", vmin=-lim, vmax=lim, shading="auto")
        ax.contour(th, z / 1e3, tm[name]["psi"][e], levels=levels, colors="k", linewidths=0.4)
        ax.plot(tm[name]["mean"][e], z / 1e3, "k", lw=2, label=f"mean {LABEL[name]}")
        ax.set_xlabel(LABEL[name] + " (K)")
        ax.set_xlim(xlim)
        ax.legend(loc="upper right")
    axs[0, 0].set_ylabel("z (km)")
    fig.colorbar(pc, ax=axs[0], label=r"$\Psi$ (kg m$^{-2}$ s$^{-1}$)")

    ax = axs[1, 0]
    for name, ls in zip(BINS, ["-", "--"]):
        ax.plot(tm[name]["M_up"][e], z / 1e3, "C0", ls=ls, label=r"$M^+$, " + LABEL[name])
        ax.plot(-tm[name]["M_down"][e], z / 1e3, "C1", ls=ls, label=r"$-M^-$, " + LABEL[name])
    ax.set_xlabel(r"mass flux (kg m$^{-2}$ s$^{-1}$)")
    ax.set_ylabel("z (km)")
    ax.legend()

    ax = axs[1, 1]
    for name, ls in zip(BINS, ["-", "--"]):
        ax.plot(np.nanmax(np.abs(tm[name]["psi"][e]), axis=-1), z / 1e3, "k", ls=ls, label=LABEL[name])
    ax.set_xlabel(r"$\max_\theta |\Psi|$ (kg m$^{-2}$ s$^{-1}$)")
    ax.legend()

    nt = int(np.isfinite(sums.MF_theta_e.isel(experiment=e, z=0, theta_e=0)).sum())
    fig.suptitle(f"{exp}: time mean over {nt} snapshots")
    plt.show()

## Save

In [ ]:
result.to_netcdf("iso_streamfunction_frozen.nc")